<a href="https://colab.research.google.com/github/Mahid-Imran/flyrank-ml-internship-mahid-assignment2/blob/main/work/notebooks/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mahid-Imran/flyrank-ml-internship-mahid-assignment2/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

I have provisionally selected **Lane 2: Refresh / Content Opportunity Scoring**. The purpose of this lane is to identify which content pages should be reviewed first for refresh, expansion, protection, pruning, or monitoring.

I selected this lane because the starter dataset contains useful observed signals such as search impressions, clicks, CTR, average position, page age, freshness, sessions, engagement, and performance trend. These signals may help identify pages that deserve earlier human review.

This lane also connects directly with the starter notebooks, where a simple rule-based method and machine learning models were used to rank pages for refresh review. My aim is not simply to train a model, but to investigate whether an understandable, evidence-based ranking can help a content team use its limited editorial time more effectively.

In [9]:
import os
import subprocess
import pandas as pd
import numpy as np

REPO_URL = "https://github.com/Mahid-Imran/fly-rank-ml-assignment2.git"
REPO_DIR = "/content/fly-rank-ml-assignment2"

# Clone the repository only if it has not already been cloned
if not os.path.exists(REPO_DIR):
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, REPO_DIR],
        check=True
    )

# Move into the repository folder
os.chdir(REPO_DIR)

# Load the starter dataset
DATA_PATH = "data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully.")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")
print(f"Unique clients: {df['client_id'].nunique():,}")

df.head(3)

Dataset loaded successfully.
Rows: 30,000
Columns: 44
Unique clients: 32


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9


## 2. The question: decision, action, cost of a wrong call

### Research question

Which content pages should be reviewed first for refresh, expansion, protection, pruning, or monitoring, based on observed search-performance, engagement, content-age, freshness, and trend signals?

### Unit of analysis

The unit of analysis is one pseudonymized content item, representing one page. Each recommendation will therefore apply to a content ID rather than a real client name or URL.

### Decision

The project will improve the decision of **which pages a content editor or SEO specialist should review first when editorial time is limited**.

### Intended output

The intended output is a ranked page-review queue. Each page should eventually have a priority score, a suggested action, understandable reason codes, and an evidence or confidence level.

### Person and action

A content editor, SEO specialist, or content manager would inspect the highest-ranked pages first. After reviewing the available evidence, the person could decide to refresh, expand, protect, merge, prune, or continue monitoring a page.

### Cost of a wrong recommendation

A false positive would place a healthy or low-value page too high in the queue. This could waste editorial time and could cause unnecessary changes to a page that did not require intervention.

A false negative would place a genuinely important page too low in the queue. The content team might then miss a meaningful decline or opportunity and continue losing potential visibility, clicks, sessions, or engagement.

### Why data or ML may help

A simple rule can provide a useful baseline, but page priority may depend on several interacting signals, including impressions, position, CTR, age, freshness, trend, sessions, and engagement. A data-based method may combine these signals more consistently.

Machine learning will only be considered useful if it performs better than a transparent baseline under honest validation and still provides explanations that a human reviewer can understand. Therefore, the main objective is decision support, not simply training a model.

In [10]:
unit_check = pd.DataFrame({
    "Measure": [
        "Total dataset rows",
        "Unique content IDs",
        "Duplicate content IDs",
        "Unique clients"
    ],
    "Value": [
        len(df),
        df["content_id"].nunique(),
        df["content_id"].duplicated().sum(),
        df["client_id"].nunique()
    ]
})

display(unit_check)

,Measure,Value
0,Total dataset rows,30000
1,Unique content IDs,30000
2,Duplicate content IDs,0
3,Unique clients,32


## 3. Quick look at the data (2-3 real numbers)

The code below examines the starter dataset and calculates several real numbers related to Refresh / Content Opportunity Scoring. I use these results to check whether the dataset contains enough potentially actionable pages to justify this lane.

In [11]:
# Pages with measurable search visibility and enough age
eligible_pages = df[
    (df["impressions_90d"] > 0) &
    (df["content_age_days"] >= 90)
].copy()

# Pages showing decline while still having meaningful search demand
declining_with_demand = df[
    (df["trend_direction"] == "down") &
    (df["impressions_90d"] >= 100)
].copy()

# Pages that are old since their last update but still visible
stale_visible_pages = df[
    (df["days_since_last_update"] >= 180) &
    (df["impressions_90d"] >= 500)
].copy()

# Pages receiving visibility but showing low CTR
low_ctr_visible_pages = df[
    (df["impressions_90d"] >= 500) &
    (df["avg_position"].between(1, 20)) &
    (df["ctr"] < 0.5)
].copy()

evidence = pd.DataFrame({
    "Measure": [
        "Total content pages",
        "Eligible pages: impressions > 0 and age >= 90 days",
        "Declining pages with at least 100 impressions",
        "Stale visible pages",
        "Low-CTR visible pages"
    ],
    "Number of pages": [
        len(df),
        len(eligible_pages),
        len(declining_with_demand),
        len(stale_visible_pages),
        len(low_ctr_visible_pages)
    ]
})

display(evidence)

print("\nPercentages:")
print(
    f"Declining with demand: "
    f"{len(declining_with_demand) / len(df) * 100:.2f}%"
)
print(
    f"Stale and visible: "
    f"{len(stale_visible_pages) / len(df) * 100:.2f}%"
)
print(
    f"Visible with low CTR: "
    f"{len(low_ctr_visible_pages) / len(df) * 100:.2f}%"
)

,Measure,Number of pages
0,Total content pages,30000
1,Eligible pages: impressions > 0 and age >= 90 ...,30000
2,Declining pages with at least 100 impressions,13152
3,Stale visible pages,17
4,Low-CTR visible pages,9745



Percentages:
Declining with demand: 43.84%
Stale and visible: 0.06%
Visible with low CTR: 32.48%


## 4. Careful words: what I can and can't claim

This project uses observational data. It records measured page and search performance, but it does not prove why a result happened.

The project may be able to report that certain measured signals are associated with decline, visibility, CTR, engagement, or review priority. It may also evaluate whether a ranking method identifies observed review candidates more effectively than a simple baseline.

The final result should be described as **decision support**. A highly ranked page means that the page deserves earlier human review based on the available evidence. It does not mean that the page is definitely poor or that editing it will definitely improve its future performance.

I will not claim that this project has discovered Google's ranking algorithm. I will not claim that a particular feature causes a page to gain or lose ranking. I will also not claim that refreshing a page guarantees recovery. Proving that an edit caused an improvement would require an experiment or another suitable causal research design.

The pseudonymized content and client IDs will only be used for grouping, checking, joining, or validation splits. They will not be used as predictive features.

If decline is used as the outcome, `trend_direction` and `trend_pct` will not be included as model features because they directly define or reveal the decline outcome.

In [12]:
print("Important leakage and interpretation checks")
print("-" * 50)

print(
    "Rows where avg_position = 0:",
    int((df["avg_position"] == 0).sum())
)

print(
    "Columns that must not be used as decline features:",
    [col for col in ["trend_direction", "trend_pct"] if col in df.columns]
)

print(
    "Identifier columns for grouping/splitting only:",
    [col for col in ["content_id", "client_id"] if col in df.columns]
)

print(
    "CTR range in the starter dataset:",
    round(df["ctr"].min(), 4),
    "to",
    round(df["ctr"].max(), 4)
)

Important leakage and interpretation checks
--------------------------------------------------
Rows where avg_position = 0: 1205
Columns that must not be used as decline features: ['trend_direction', 'trend_pct']
Identifier columns for grouping/splitting only: ['content_id', 'client_id']
CTR range in the starter dataset: 0.0 to 100.0


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.